# Qwen Training Agent — Google Colab

This notebook connects to your Qwen Training Engine server and uses Colab's GPU for LoRA fine-tuning.

**Flow:** Server generates training data (Gemini) → sends to this agent via WebSocket → agent trains LoRA on Colab GPU → uploads adapter back to server.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies & Clone Repo

In [ ]:
!pip install -q websockets httpx pynvml psutil \
    torch transformers peft bitsandbytes datasets accelerate

!rm -rf /content/qwen-training
!git clone https://github.com/kmalarifi97/qwen-training.git /content/qwen-training
!ls /content/qwen-training/agent/

## 3. Configure & Start Agent

In [ ]:
import json, os, subprocess, sys

# Write config
config = {
    "server_url": "ws://34.18.164.66:8090/agents/connect",
    "agent_name": "colab-gpu",
    "api_key": "",
    "heartbeat_interval_seconds": 10,
    "model_cache_dir": "/content/models",
    "log_dir": "/content/logs",
}

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/logs", exist_ok=True)

with open("/content/qwen-training/agent/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config written. Starting agent...")
print(json.dumps(config, indent=2))
print()

# Start agent
os.chdir("/content/qwen-training/agent")

process = subprocess.Popen(
    [sys.executable, "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    process.terminate()
    print("\n--- Agent stopped ---")